In [1]:
%load_ext autoreload
%autoreload 2
import warnings
from pandas.errors import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
from app.logger import *
import json5,json
import fitz #type: ignore

from app.insur.fund_data import *
from app.utils import *
from app.konstant import get_config, get_regex

from app.amc.fund_data import *

utils = Helper()

In [2]:
#INSURANCE FUND
amc_id = '81_0'
path = r"81_30-Apr-26_IF.pdf"
config = get_config("2026",amc_id)
regex = get_regex("2026")

object = GeneraliLifeINSR(config,regex,path)
title,path_pdf= object.check_and_highlight(path)
# print("done")
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

[GeneraliLifeINSR._get_normal_title] (81_30-Apr-26_IF.pdf) FileNotFoundError: no such file: '81_30-Apr-26_IF.pdf'
Traceback (most recent call last):
  File "c:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\rep_fsparse\app\logger.py", line 146, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\rep_fsparse\app\insur\parse_pdf.py", line 52, in _get_normal_title
    with fitz.open(path) as doc:
         ^^^^^^^^^^^^^^^
  File "c:\Users\kaustubh.keny\AppData\Local\Programs\Python\Python312\Lib\site-packages\pymupdf\__init__.py", line 2969, in __init__
    raise FileNotFoundError(f"no such file: '{filename}'")
pymupdf.FileNotFoundError: no such file: '81_30-Apr-26_IF.pdf'

[GeneraliLifeINSR.check_and_highlight] (81_30-Apr-26_IF.pdf) FileNotFoundError: no such file: '81_30-Apr-26_IF.pdf'
Traceback (most recent call last):
  File "c:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\rep_fsparse\app\logger.py", line 14

C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\rep_fsparse\config\2026\81_0_AMC.json5


TypeError: cannot unpack non-iterable NoneType object

In [ ]:
# object = GeneraliLifeINSR(config,regex,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)


In [ ]:
save_path = os.path.join(object.JSON_PATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

In [ ]:
pattern = "([A-Z]{1}[a-z]+\\s*[A-Z]{1}[a-z]+)"
for fund, content in final_text.items():
    # check = 'before.fund_manager'
    for key in content:
        if key.endswith("fund_manager_details"):
            print(fund)
            text =re.sub("[^A-Za-z0-9\\s\\-\\(\\)\\.\\,\\+\\%\\:\\&]+", "",content[key]).strip()
            print(text)
            match = re.findall(pattern,text, re.IGNORECASE)
            print(match)

In [2]:
import json
import csv
from typing import Union, Dict, Any, List


def json_to_csv(input_data: str,output_file: str,spacing: int = 2) -> int:
    """
    Convert mutual fund JSON into a structured CSV.

    Parameters:
        output_file (str): Path to output CSV file
        spacing (int): Number of empty rows after each mutual fund

    Returns:
        int: Total number of rows written (excluding header)

    Raises:
        ValueError: If input data format is invalid
        IOError: If file operations fail
    """
    try:
        with open(input_data, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        raise



    # Validate
    if "records" not in data or not isinstance(data["records"], list):
        raise ValueError("Invalid JSON structure")

    row_count = 0

    try:
        with open(output_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)

            # Header
            writer.writerow([
                "mutual_fund_name",
                "main_scheme_name",
                "monthly_aaum_value",
                "empty_col",
                "table",
                "portfolio_data_0",
                "portfolio_data_1"
            ])

            # Process each record
            for record in data["records"]:
                value = record.get("value", {})

                mf_name = value.get("mutual_fund_name", "")
                scheme_name = value.get("main_scheme_name", "")
                aum_value = value.get("monthly_aaum_value", "")

                portfolio_list: List[Dict[str, Any]] = value.get("portfolio_data", [])

                # Skip if no portfolio data
                if not isinstance(portfolio_list, list):
                    continue

                for item in portfolio_list:
                    writer.writerow([
                        mf_name,
                        scheme_name,
                        aum_value,
                        "",
                        item.get("table", ""),
                        item.get("0", ""),
                        item.get("1", "")
                    ])
                    row_count += 1

                # Add spacing rows
                for _ in range(spacing):
                    writer.writerow([])

        return row_count

    except Exception as e:
        raise IOError(f"Error writing CSV: {e}")

In [3]:
path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\81_30-Apr-26_IF.json"
output_path = r"IF.csv"
json_to_csv(path,output_path)

481